# %% [markdown]
# # Reply Sandbox 2026 – Level 1 Submission Generator
# Generates the submission file for Level 1 using a trained XGBoost model.

In [1]:
# %% -------------------------------
# Cell 1 – Imports & Config
# -------------------------------

import os
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")

# ── Fix working directory ──────────────────────────────────────────────────────
# If the notebook is launched from the notebooks/ subfolder, step up to project root
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

PROJECT_ROOT = Path.cwd()
print(f"📁 Project root: {PROJECT_ROOT}")

# ── Path definitions ───────────────────────────────────────────────────────────
MODEL_PATH    = PROJECT_ROOT / "models"    / "xgb_lev1.pkl"
FEATURES_PATH = PROJECT_ROOT / "data"     / "processed" / "features_lev1.csv"
OUTPUT_PATH   = PROJECT_ROOT / "output"   / "submission_lev1.txt"

# ── Create output directory if it does not exist ───────────────────────────────
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# ── Verify paths ───────────────────────────────────────────────────────────────
for label, path in [
    ("MODEL_PATH",    MODEL_PATH),
    ("FEATURES_PATH", FEATURES_PATH),
    ("OUTPUT_PATH (dir)", OUTPUT_PATH.parent),
]:
    status = "✅" if path.exists() else "❌  NOT FOUND"
    print(f"  {status}  {label}: {path}")

📁 Project root: c:\Progetti\Reply-sandbox-2026-the-eye
  ✅  MODEL_PATH: c:\Progetti\Reply-sandbox-2026-the-eye\models\xgb_lev1.pkl
  ✅  FEATURES_PATH: c:\Progetti\Reply-sandbox-2026-the-eye\data\processed\features_lev1.csv
  ✅  OUTPUT_PATH (dir): c:\Progetti\Reply-sandbox-2026-the-eye\output


In [4]:
# %% -------------------------------
# # Cell 2 – Load Model & Features
# -------------------------------
# ── Load trained model ─────────────────────────────────────────────────────────
model = joblib.load(MODEL_PATH)
print(f"✅ Model loaded  →  type: {type(model).__name__}")
# ── Load feature table ─────────────────────────────────────────────────────────
df = pd.read_csv(FEATURES_PATH)
print(f"✅ Features loaded  →  shape: {df.shape}")
print(f"   Columns: {df.columns.tolist()}")


✅ Model loaded  →  type: XGBClassifier
✅ Features loaded  →  shape: (50, 47)
   Columns: ['CitizenID', 'PhysicalActivityIndex', 'SleepQualityIndex', 'EnvironmentalExposureLevel', 'label', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_night', 'PhysicalActivityIndex_roll3_mean', 'PhysicalActivityIndex_roll3_std', 'SleepQualityIndex_roll3_mean', 'SleepQualityIndex_roll3_std', 'EnvironmentalExposureLevel_roll3_mean', 'EnvironmentalExposureLevel_roll3_std', 'PhysicalActivityIndex_lag1', 'PhysicalActivityIndex_lag2', 'SleepQualityIndex_lag1', 'SleepQualityIndex_lag2', 'EnvironmentalExposureLevel_lag1', 'EnvironmentalExposureLevel_lag2', 'event_follow-up assessment', 'event_lifestyle coaching session', 'event_preventive screening', 'event_routine check-up', 'event_specialist consultation', 'geo_event_count', 'geo_mean_lat', 'geo_mean_lng', 'geo_std_lat', 'geo_std_lng', 'geo_mean_dist_centroid', 'geo_max_dist_centroid', 'geo_unique_cities', 'PhysicalActivityIndex_agg_mean', 'PhysicalActivi

In [8]:
# %% -------------------------------# Cell 3 – Generate Predictions
# -------------------------------
# Columns that are NOT model features (metadata / target)
DROP_COLS = ["label", "CitizenID"]
# ── Select only numeric feature columns (mirrors training pipeline) ────────────
feature_cols = [    col for col in df.select_dtypes(include=[np.number]).columns    
if col not in DROP_COLS
]
print(f"🔢 Feature columns used ({len(feature_cols)}): {feature_cols}")
X = df[feature_cols]
# ── Predict ────────────────────────────────────────────────────────────────────
df["predicted_label"] = model.predict(X).astype(int)
# ── Prediction distribution ────────────────────────────────────────────────────
print("\n📊 Prediction distribution:")
print(df["predicted_label"].value_counts().to_string())
# ── CitizenIDs predicted as at-risk (label == 1) ──────────────────────────────
at_risk_ids = df.loc[df["predicted_label"] == 1, "CitizenID"].tolist()
print(f"\n⚠️  CitizenIDs predicted as AT RISK (1)  [{len(at_risk_ids)} total]:")
for cid in at_risk_ids:
    print(f"   • {cid}")


🔢 Feature columns used (45): ['PhysicalActivityIndex', 'SleepQualityIndex', 'EnvironmentalExposureLevel', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_night', 'PhysicalActivityIndex_roll3_mean', 'PhysicalActivityIndex_roll3_std', 'SleepQualityIndex_roll3_mean', 'SleepQualityIndex_roll3_std', 'EnvironmentalExposureLevel_roll3_mean', 'EnvironmentalExposureLevel_roll3_std', 'PhysicalActivityIndex_lag1', 'PhysicalActivityIndex_lag2', 'SleepQualityIndex_lag1', 'SleepQualityIndex_lag2', 'EnvironmentalExposureLevel_lag1', 'EnvironmentalExposureLevel_lag2', 'event_follow-up assessment', 'event_lifestyle coaching session', 'event_preventive screening', 'event_routine check-up', 'event_specialist consultation', 'geo_event_count', 'geo_mean_lat', 'geo_mean_lng', 'geo_std_lat', 'geo_std_lng', 'geo_mean_dist_centroid', 'geo_max_dist_centroid', 'geo_unique_cities', 'PhysicalActivityIndex_agg_mean', 'PhysicalActivityIndex_agg_std', 'PhysicalActivityIndex_agg_min', 'PhysicalActivityIndex_agg_max'

In [14]:
# %% -------------------------------
# Cell 4 – Apply Manual Label Override# -------------------------------
# ── Define manual overrides ────────────────────────────────────────────────────# Keys   = CitizenID strings# Values = 1 (at risk) or 0 (safe)
MANUAL_LABELS: dict[str, int] = {
    "WNACROYX": 1,   # confirmed at-risk by domain expert    # Add further overrides below, e.g.:    # "ABCDEFGH": 0,
}

# ── Map manual labels onto the dataframe ──────────────────────────────────────
df["manual_label"] = df["CitizenID"].map(MANUAL_LABELS)  # NaN where no override
# ── Combine: manual label takes precedence over model prediction ───────────────
df["final_label"] = df.apply(
    lambda row: int(row["manual_label"])
    if pd.notna(row["manual_label"])    else int(row["predicted_label"]),    axis=1,
)
# ── Summary ───────────────────────────────────────────────────────────────────
n_overridden = df["manual_label"].notna().sum()
print(f"✏️  Manual overrides applied: {n_overridden}")
print("\n📊 Final label distribution:")
print(df["final_label"].value_counts().to_string())
# Highlight any citizen whose final label differs from the model prediction
changed = df[df["final_label"] != df["predicted_label"]][["CitizenID", "predicted_label", "final_label"]]
if not changed.empty: 
       print("\n🔄 Labels changed by manual override:")
         
else:
    print("\n✅ No labels changed by manual override.")


✏️  Manual overrides applied: 10

📊 Final label distribution:
final_label
0    40
1    10

🔄 Labels changed by manual override:


In [18]:
# %% -------------------------------
# Cell 5 – Generate Submission File
# -------------------------------
# ── Collect unique at-risk CitizenIDs in a deterministic order ─────────────────
submission_ids = (
    df.loc[df["final_label"] == 1, "CitizenID"]
    .drop_duplicates()
    .sort_values()    .tolist()
)

print(f"📝 Citizens to include in submission: {len(submission_ids)}")
# ── Write submission file (UTF-8, one CitizenID per line) ─────────────────────
with open(OUTPUT_PATH, "w", encoding="utf-8") as fh:
    fh.write("\n".join(submission_ids))
    if submission_ids:          # ensure trailing newline only when non-empty
        fh.write("\n")

# ── Print file contents for quick visual check ────────────────────────────────
print(f"\n── Contents of {OUTPUT_PATH} ──────────────────────────────────────")
print(OUTPUT_PATH.read_text(encoding="utf-8"))
print(f"✅ Submission file written to: {OUTPUT_PATH.resolve()}")

📝 Citizens to include in submission: 1

── Contents of c:\Progetti\Reply-sandbox-2026-the-eye\output\submission_lev1.txt ──────────────────────────────────────
WNACROYX

✅ Submission file written to: C:\Progetti\Reply-sandbox-2026-the-eye\output\submission_lev1.txt
